In [34]:
# 1. 데이터셋 다운로드
import os
import urllib.request
import zipfile

os.makedirs('/content', exist_ok=True)
urllib.request.urlretrieve("http://files.grouplens.org/datasets/movielens/ml-100k.zip", "/content/ml-100k.zip")

with zipfile.ZipFile("/content/ml-100k.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [35]:
# 2. 라이브러리 임포트
import math
import random
import pickle
from collections import defaultdict
from pathlib import Path
import numpy as np

In [36]:
# 3. 데이터 로드 함수
def read_ratings(path):
    rows = []
    with open(path, 'r', encoding='latin-1') as f:
        for line in f:
            user_id, item_id, rating, timestamp = line.strip().split('\t')
            rows.append((int(user_id), int(item_id), float(rating), int(timestamp)))
    return rows

def read_items(path):
    titles = {}
    genres = {}
    with open(path, 'r', encoding='latin-1') as f:
        for line in f:
            parts = line.rstrip('\n').split('|')
            iid = int(parts[0])
            titles[iid] = parts[1]
            genre_vec = np.array([int(x) for x in parts[5:24]], dtype=np.float32)
            norm = np.linalg.norm(genre_vec)
            genres[iid] = genre_vec / norm if norm > 0 else genre_vec
    return titles, genres

In [37]:
# 4. ItemKNNRecommender 클래스
class ItemKNNRecommender:
    def __init__(self, k=40, shrinkage=25.0, min_candidate_popularity=5):
        self.k = k
        self.shrinkage = shrinkage
        self.min_candidate_popularity = min_candidate_popularity

    def fit(self, rows):
        self.users = sorted({u for u, _, _, _ in rows})
        self.items = sorted({i for _, i, _, _ in rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.ratings = np.full((n_users, n_items), np.nan, dtype=np.float32)
        self.user_rated_items = defaultdict(list)
        self.item_popularity = defaultdict(int)
        for user_id, item_id, rating, _ in rows:
            uidx = self.user_to_idx[user_id]
            iidx = self.item_to_idx[item_id]
            self.ratings[uidx, iidx] = rating
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
        self.global_mean = float(np.nanmean(self.ratings))
        self.user_means = np.nanmean(self.ratings, axis=1)
        self.item_means = np.nanmean(self.ratings, axis=0)
        self.user_means = np.where(np.isnan(self.user_means), self.global_mean, self.user_means)
        self.item_means = np.where(np.isnan(self.item_means), self.global_mean, self.item_means)
        mask = ~np.isnan(self.ratings)
        centered = np.where(mask, self.ratings - self.user_means[:, None], 0.0)
        numerator = centered.T @ centered
        norms = np.sqrt(np.sum(centered * centered, axis=0))
        denominator = norms[:, None] * norms[None, :]
        similarity = np.divide(numerator, denominator,
            out=np.zeros_like(numerator, dtype=np.float32), where=denominator > 0)
        co_counts = mask.astype(np.float32).T @ mask.astype(np.float32)
        significance = co_counts / (co_counts + self.shrinkage)
        self.similarity = similarity * significance
        np.fill_diagonal(self.similarity, 0.0)
        self.neighbors = {}
        for item_idx in range(n_items):
            sims = self.similarity[item_idx]
            order = np.argsort(np.abs(sims))[::-1]
            order = [idx for idx in order if sims[idx] != 0.0][: self.k]
            self.neighbors[item_idx] = order
        return self

    def predict(self, user_id, item_id):
        if user_id not in self.user_to_idx:
            return self._clip(self.item_mean(item_id))
        if item_id not in self.item_to_idx:
            return self._clip(self.user_mean(user_id))
        uidx = self.user_to_idx[user_id]
        target_idx = self.item_to_idx[item_id]
        baseline = self.user_means[uidx]
        numerator = denominator = 0.0
        for neighbor_idx in self.neighbors[target_idx]:
            rating = self.ratings[uidx, neighbor_idx]
            if np.isnan(rating):
                continue
            sim = float(self.similarity[target_idx, neighbor_idx])
            numerator += sim * (float(rating) - baseline)
            denominator += abs(sim)
        if denominator == 0.0:
            return self._clip(0.6 * baseline + 0.4 * self.item_means[target_idx])
        return self._clip(baseline + numerator / denominator)

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {item_id for item_id, _ in self.user_rated_items[user_id]}
        candidates = [
            item_id for item_id in self.items
            if item_id not in seen
            and self.item_popularity.get(item_id, 0) >= self.min_candidate_popularity
        ]
        scored = [(item_id, self.predict(user_id, item_id)) for item_id in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n):
        return [(item_id, self.item_mean(item_id))
            for item_id, _ in sorted(self.item_popularity.items(),
                key=lambda x: (x[1], self.item_mean(x[0])), reverse=True)[:top_n]]

    def user_mean(self, user_id):
        if user_id not in self.user_to_idx:
            return self.global_mean
        return float(self.user_means[self.user_to_idx[user_id]])

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx:
            return self.global_mean
        return float(self.item_means[self.item_to_idx[item_id]])

    @staticmethod
    def _clip(value): return min(5.0, max(1.0, float(value)))

In [38]:
# 5. MFRecommender 클래스
class MFRecommender:
    def __init__(self, n_factors=20, lr=0.005, reg=0.02, n_epochs=30,
                 min_candidate_popularity=1, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_candidate_popularity = min_candidate_popularity
        self.seed = seed

    def fit(self, train_rows):
        self.users = sorted({u for u, _, _, _ in train_rows})
        self.items = sorted({i for _, i, _, _ in train_rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)
        self.user_rated_items = defaultdict(list)
        self.item_popularity  = defaultdict(int)
        rating_sum = 0.0
        for user_id, item_id, rating, _ in train_rows:
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
            rating_sum += rating
        self.global_mean = rating_sum / len(train_rows) if train_rows else 3.0
        rng = np.random.default_rng(self.seed)
        self.U   = rng.normal(0, 0.01, (n_users, self.n_factors)).astype(np.float32)
        self.V   = rng.normal(0, 0.01, (n_items, self.n_factors)).astype(np.float32)
        self.b_u = np.zeros(n_users, dtype=np.float32)
        self.b_i = np.zeros(n_items, dtype=np.float32)
        train_list = [(self.user_to_idx[u], self.item_to_idx[i], r) for u, i, r, _ in train_rows]
        rng_shuffle = random.Random(self.seed)
        for epoch in range(self.n_epochs):
            rng_shuffle.shuffle(train_list)
            for uidx, iidx, rating in train_list:
                pred = (self.global_mean + self.b_u[uidx] + self.b_i[iidx] + float(self.U[uidx] @ self.V[iidx]))
                err    = rating - pred
                v_old = self.V[iidx].copy()
                self.V[iidx]   += self.lr * (err * self.U[uidx] - self.reg * self.V[iidx])
                self.U[uidx]   += self.lr * (err * v_old        - self.reg * self.U[uidx])
                self.b_u[uidx] += self.lr * (err - self.reg * self.b_u[uidx])
                self.b_i[iidx] += self.lr * (err - self.reg * self.b_i[iidx])
        return self

    def predict(self, user_id, item_id):
        has_u = user_id in self.user_to_idx
        has_i = item_id in self.item_to_idx
        if not has_u and not has_i: return self._clip(self.global_mean)
        if not has_u: return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])
        if not has_i: return self._clip(self.global_mean + self.b_u[self.user_to_idx[user_id]])
        uidx, iidx = self.user_to_idx[user_id], self.item_to_idx[item_id]
        return self._clip(self.global_mean + self.b_u[uidx] + self.b_i[iidx] + float(self.U[uidx] @ self.V[iidx]))

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {iid for iid, _ in self.user_rated_items[user_id]}
        candidates = [iid for iid in self.items if iid not in seen and self.item_popularity.get(iid, 0) >= self.min_candidate_popularity]
        scored = [(iid, self.predict(user_id, iid)) for iid in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n=10):
        sorted_items = sorted(self.item_popularity.keys(), key=lambda iid: (self.item_popularity[iid], self.item_mean(iid)), reverse=True)
        return [(iid, self.item_mean(iid)) for iid in sorted_items[:top_n]]

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx: return self.global_mean
        return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])

    @staticmethod
    def _clip(value): return min(5.0, max(1.0, float(value)))

In [39]:
# 6. EnsembleRecommender 클래스
class EnsembleRecommender:
    def __init__(self, knn_model, mf_model, weight_knn=0.5, min_candidate_popularity=1):
        self.knn = knn_model
        self.mf  = mf_model
        self.w_knn = weight_knn
        self.w_mf  = 1.0 - weight_knn
        self.min_candidate_popularity = min_candidate_popularity
        self.items = sorted(set(knn_model.items) | set(mf_model.items))
        self.item_popularity = mf_model.item_popularity
        self.user_rated_items = defaultdict(set)
        for uid, rated in knn_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        for uid, rated in mf_model.user_rated_items.items():
            for item_id, _ in rated:
                self.user_rated_items[uid].add(item_id)
        self.users = sorted(set(knn_model.users) | set(mf_model.users))
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}

    def predict(self, user_id, item_id):
        return min(5.0, max(1.0, self.w_knn * self.knn.predict(user_id, item_id) + self.w_mf * self.mf.predict(user_id, item_id)))

    def recommend(self, user_id, top_n=10):
        seen = self.user_rated_items.get(user_id, set())
        candidates = [iid for iid in self.items if iid not in seen and self.item_popularity.get(iid, 0) >= self.min_candidate_popularity]
        scored = [(iid, self.predict(user_id, iid)) for iid in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

In [40]:
# 7. 평가
def rating_metrics(model, rows):
    sq_errors, abs_errors = [], []
    for user_id, item_id, rating, _ in rows:
        pred = model.predict(user_id, item_id)
        e = rating - pred
        sq_errors.append(e * e)
        abs_errors.append(abs(e))
    return {
        'rmse': math.sqrt(sum(sq_errors) / len(sq_errors)),
        'mae':  sum(abs_errors) / len(abs_errors),
    }

def topn_metrics(model, eval_rows, genres, top_n=10, relevant_threshold=4.0):
    eval_users = sorted({u for u, _, _, _ in eval_rows if u in model.user_to_idx})
    relevant_by_user = defaultdict(set)
    for user_id, item_id, rating, _ in eval_rows:
        if rating >= relevant_threshold:
            relevant_by_user[user_id].add(item_id)
            
    all_recommended = set()
    novelty_scores, diversity_scores = [], []
    precisions, recalls, f1_scores = [], [], []
    total_interactions = sum(model.item_popularity.values())
    
    for user_id in eval_users:
        recs      = model.recommend(user_id, top_n=top_n)
        rec_items = [iid for iid, _ in recs]
        all_recommended.update(rec_items)
        
        # Novelty 계산
        for iid in rec_items:
            pop = max(model.item_popularity.get(iid, 1), 1)
            novelty_scores.append(-math.log2(pop / total_interactions))
            
        # Diversity 계산 
        if len(rec_items) >= 2:
            vecs = [genres[iid] for iid in rec_items if iid in genres]
            if len(vecs) >= 2:
                sim_sum, pair_count = 0.0, 0
                for a in range(len(vecs)):
                    for b in range(a + 1, len(vecs)):
                        sim_sum    += float(np.dot(vecs[a], vecs[b]))
                        pair_count += 1
                avg_sim = sim_sum / pair_count if pair_count > 0 else 0.0
                diversity_scores.append(1.0 - avg_sim) 
                
        # Precision,Recall,F1
        relevant = relevant_by_user.get(user_id, set())
        if relevant:
            hits = len(set(rec_items) & relevant)
            p = hits / top_n
            r = hits / len(relevant)
            precisions.append(p)
            recalls.append(r)
            if (p + r) > 0:
                f1_scores.append(2 * p * r / (p + r))
            else:
                f1_scores.append(0.0)
                
    return {
        'precision': np.mean(precisions) if precisions else 0.0,
        'recall':    np.mean(recalls) if recalls else 0.0,
        'f1_score':  np.mean(f1_scores) if f1_scores else 0.0,
        'novelty':   np.mean(novelty_scores) if novelty_scores else 0.0,
        'diversity': np.mean(diversity_scores) if diversity_scores else 0.0
    }

In [41]:
# 8. 저장된 모델 로드, 평가 실행

import os

DATA_DIR = Path('/content/ml-100k')
test_rows = read_ratings(DATA_DIR / 'ua.test')
titles, genres = read_items(DATA_DIR / 'u.item')

save_path = 'saved_models/ensemble_model.pkl'

if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        bundle = pickle.load(f)
    
    
    knn_model = bundle['knn']
    mf_model = bundle['mf']
    ensemble_model = bundle['ensemble']
    
    print("=" * 60)
    print(f"   * 학습된 최적 Item-KNN 이웃 수 (K): {bundle.get('best_k', 'N/A')}")
    print(f"   * 학습된 최적 MF 잠재요인 수 (Factors): {bundle.get('best_factors', 'N/A')}")
    print(f"   * 학습된 최적 KNN 결합 가중치: {bundle.get('best_w_knn', 0.5):.2f}")
    print("=" * 60)
    
    #Item-KNN 모델 평가
    knn_r = rating_metrics(knn_model, test_rows)
    knn_t = topn_metrics(knn_model, test_rows, genres, top_n=10)
    
    #Matrix Factorization 모델 평가
    mf_r = rating_metrics(mf_model, test_rows)
    mf_t = topn_metrics(mf_model, test_rows, genres, top_n=10)
    
    #가중치 앙상블 모델 평가
    ens_r = rating_metrics(ensemble_model, test_rows)
    ens_t = topn_metrics(ensemble_model, test_rows, genres, top_n=10)
    
  
    print("\n Item-Based Collaborative Filtering 결과")
    print(f"  - RMSE (예측 오차)   : {knn_r['rmse']:.4f}  | MAE (예측 오차)   : {knn_r['mae']:.4f}")
    print(f"  - Precision@10       : {knn_t['precision']:.4f}  | Recall@10       : {knn_t['recall']:.4f}")
    print(f"  - Novelty (신선도)   : {knn_t['novelty']:.4f}  | Diversity (다양성): {knn_t['diversity']:.4f}")
    
    print("\n Matrix Factorization 결과")
    print(f"  - RMSE (예측 오차)   : {mf_r['rmse']:.4f}  | MAE (예측 오차)   : {mf_r['mae']:.4f}")
    print(f"  - Precision@10       : {mf_t['precision']:.4f}  | Recall@10       : {mf_t['recall']:.4f}")
    print(f"  - Novelty (신선도)   : {mf_t['novelty']:.4f}  | Diversity (다양성): {mf_t['diversity']:.4f}")
    
    print("\n 최종 가중치 합산 앙상블(Ensemble) 결과")
    print(f"  - RMSE (예측 오차)   : {ens_r['rmse']:.4f}  | MAE (예측 오차)   : {ens_r['mae']:.4f}")
    print(f"  - Precision@10       : {ens_t['precision']:.4f}  | Recall@10       : {ens_t['recall']:.4f}")
    print(f"  - Novelty (신선도)   : {ens_t['novelty']:.4f}  | Diversity (다양성): {ens_t['diversity']:.4f}")
    print("=" * 60)
else:
    print(f"오류")

   * 학습된 최적 Item-KNN 이웃 수 (K): 80
   * 학습된 최적 MF 잠재요인 수 (Factors): 100
   * 학습된 최적 KNN 결합 가중치: 0.10

 Item-Based Collaborative Filtering 결과
  - RMSE (예측 오차)   : 0.9867  | MAE (예측 오차)   : 0.7655
  - Precision@10       : 0.0073  | Recall@10       : 0.0114
  - Novelty (신선도)   : 11.7576  | Diversity (다양성): 0.7072

 Matrix Factorization 결과
  - RMSE (예측 오차)   : 0.9422  | MAE (예측 오차)   : 0.7438
  - Precision@10       : 0.0315  | Recall@10       : 0.0541
  - Novelty (신선도)   : 9.5344  | Diversity (다양성): 0.7387

 최종 가중치 합산 앙상블(Ensemble) 결과
  - RMSE (예측 오차)   : 0.9375  | MAE (예측 오차)   : 0.7398
  - Precision@10       : 0.0318  | Recall@10       : 0.0552
  - Novelty (신선도)   : 9.5170  | Diversity (다양성): 0.7399
